# tp-17 欠損値のモデル補完・外れ値の変換（NN と LightGBM）

**問い**: 欠損値を「他の特徴量から予測した値」で埋めてから学習するテクニックは効くか。現状は、GBDT は欠損をそのまま扱い、NN は中央値で埋めて欠損フラグを足し、z スコアを ±5 で切っている（モデルで埋める処理は無い）。

## 比較（try both）
| 変種 | 内容 |
|------|------|
| 基準 | 現行（NN: 中央値埋め + 欠損フラグ + ±5 clip / LightGBM: 欠損そのまま） |
| soft8 / soft16 | 低ランク（8 / 16 成分）の反復補完（Soft-Impute 系）+ 欠損フラグを保持 |
| rg | RankGauss（分位点で正規分布へ。外れ値は端に張り付く）。欠損は従来どおり |
| rg+soft8 | RankGauss の後、欠損を低ランク補完 |

補完・変換の学習（平均・分散・主成分・分位点）は **FIT_END = 2021-12-21 以前の行だけ**で行い、全期間に適用する（行ごとの計算で、未来の情報は入らない）。
NN は 3 seed、LightGBM は 2 seed、valid 2022〜2024。採用基準 = 勝率 LogLoss の差の 95% 信頼区間の上限 < 0、効果 ≥ 0.0002、連対・複勝が 0.0002 を超えて悪化しない。test 2025 は使わない。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "4")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, gc, time
import numpy as np
import pandas as pd
import mlflow

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import tp05_data, zoo, zoo_runner, feature_bakeoff as fb
from src.pipeline.models.compute import detect_compute
from src.pipeline.features import imputation as im

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
OUT = WORK / "improve_i"
OUT.mkdir(parents=True, exist_ok=True)
THREADS = int(os.environ.get("TP17_THREADS", detect_compute(reserve_cores=6).threads))
SMOKE = os.environ.get("TP17_SMOKE") == "1"
mlflow.set_experiment("keiba-tp-imputation")
require_tracking_server()
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
D = tp05_data.load_dataset(WORK)
df, meta, folds = D["df"], D["meta"], D["folds"]
EP, EXTRAS, PED = tp05_data.load_extras_ped(df, WORK)
X0 = pd.concat([df[D["FEATS"]["base+S+C1"]], EP], axis=1)
best = json.loads((WORK / "hpo/best_params.json").read_text(encoding="utf-8"))["best"]
LP, NP = zoo.tuned_params("lgb", best["lgb"]["params"]), zoo.tuned_params("nn", best["nn"]["params"])
if SMOKE:
    folds = folds[:1]
fit_mask = (pd.to_datetime(df["date"]).to_numpy() <= np.datetime64("2021-12-21"))
num_cols = [c for c in X0.columns if str(X0[c].dtype).startswith(("float", "int")) and X0[c].nunique() > 20]
nan_cols = [c for c in num_cols if X0[c].isna().any()]
print("行", len(X0), "| 数値列", len(num_cols), "| 欠損のある数値列", len(nan_cols), "| fit 行", int(fit_mask.sum()))

## 補完・変換した特徴量表を作る

In [ ]:
key = ["race_id", "horse_id"]
fin_map = meta.set_index(key)["finish_position"]
VAR = {}
t0 = time.time()
VAR["soft8"] = im.impute_frame(X0, fit_mask, nan_cols, rank=8)
VAR["soft16"] = im.impute_frame(X0, fit_mask, nan_cols, rank=16)
VAR["rg"] = im.rank_gauss(X0, fit_mask, num_cols)
VAR["rg+soft8"] = im.impute_frame(VAR["rg"], fit_mask, nan_cols, rank=8)
print(f"作成 {time.time() - t0:.0f}s |", {k: v.shape[1] for k, v in VAR.items()})
# 補完の精度（観測値をわざと隠して戻せるか）: fit 期間の行で 10% の観測値を隠す
A = X0[nan_cols].to_numpy(dtype="float64")[fit_mask]
rng = np.random.default_rng(0)
obs = ~np.isnan(A)
hide = obs & (rng.random(A.shape) < 0.1)
Ah = np.where(hide, np.nan, A)
rows = []
for k in (4, 8, 16):
    B = im.SoftImputer(rank=k).fit(Ah).transform(Ah)
    sd = np.nanstd(A, axis=0)
    rows.append({"方法": f"低ランク rank={k}", "RMSE/sd": float(np.sqrt(np.nanmean(((B - A)[hide] / np.broadcast_to(sd, A.shape)[hide]) ** 2)))})
med = np.nanmedian(Ah, axis=0)
rows.append({"方法": "中央値", "RMSE/sd": float(np.sqrt(np.nanmean(((np.broadcast_to(med, A.shape) - A)[hide] / np.broadcast_to(sd, A.shape)[hide]) ** 2)))})
HID = pd.DataFrame(rows); display(HID.round(4))

## NN: 基準と各変種（3 seed）

In [ ]:
SEEDS_NN = [0, 1, 2]
ROWS = []
resB, OOFB = zoo_runner.run_oof("nn_imp_base", "nn", X0, meta, folds, SEEDS_NN, THREADS, params=dict(NP))
print(f"NN 基準: win {resB['logloss_win'].mean():.5f} top2 {resB['logloss_top2'].mean():.5f} top3 {resB['logloss_top3'].mean():.5f}", flush=True)
names = list(VAR) if not SMOKE else ["soft8"]
for name in names:
    t0 = time.time()
    res, oof = zoo_runner.run_oof(f"nn_imp_{name}", "nn", VAR[name], meta, folds, SEEDS_NN, THREADS, params=dict(NP))
    r = fb.compare_oof(OOFB, oof, fin_map, f"NN {name}")
    ROWS.append(r)
    print(f"NN {name}: win {res['logloss_win'].mean():.5f} Δ {r['Δwin']:+.5f} (上限 {r['win_hi']:+.5f}) top2 {r['Δtop2']:+.5f} top3 {r['Δtop3']:+.5f} 採用={r['採用']} ({time.time() - t0:.0f}s)", flush=True)
    pd.DataFrame(ROWS).set_index("variant").to_csv(OUT / "imputation_vs_base.csv")
    gc.collect()

## LightGBM: 基準（欠損そのまま）と補完版（2 seed）

In [ ]:
SEEDS_L = [0, 1]
resL, OOFL = zoo_runner.run_oof("lgb_imp_base", "lgb_binary", X0, meta, folds, SEEDS_L, THREADS, params=dict(LP))
print(f"LightGBM 基準: win {resL['logloss_win'].mean():.5f}", flush=True)
for name in (["soft8", "rg"] if not SMOKE else ["soft8"]):
    t0 = time.time()
    res, oof = zoo_runner.run_oof(f"lgb_imp_{name}", "lgb_binary", VAR[name], meta, folds, SEEDS_L, THREADS, params=dict(LP))
    r = fb.compare_oof(OOFL, oof, fin_map, f"LightGBM {name}")
    ROWS.append(r)
    print(f"LightGBM {name}: win {res['logloss_win'].mean():.5f} Δ {r['Δwin']:+.5f} (上限 {r['win_hi']:+.5f}) top2 {r['Δtop2']:+.5f} top3 {r['Δtop3']:+.5f} 採用={r['採用']} ({time.time() - t0:.0f}s)", flush=True)
    pd.DataFrame(ROWS).set_index("variant").to_csv(OUT / "imputation_vs_base.csv")
R = pd.DataFrame(ROWS).set_index("variant")
display(R[["win_base", "win_new", "Δwin", "win_hi", "Δtop2", "Δtop3", "採用"]].round(5))
adopted = list(R.index[R["採用"]])
(OUT / "report.json").write_text(json.dumps({"git_commit": GIT_COMMIT, "results": R.round(6).reset_index().to_dict("records"), "hidden_value_recovery": HID.round(5).to_dict("records"), "adopted": adopted, "n_nan_cols": len(nan_cols)},
                                            ensure_ascii=False, indent=1, default=str), encoding="utf-8")
with mlflow.start_run(run_name="tp-17-imputation", tags={"notebook": "tp-17", "git_commit": GIT_COMMIT}):
    for i_, n in enumerate(R.index):
        mlflow.log_metrics({f"delta_win_{i_:02d}": float(R.loc[n, "Δwin"])})
    mlflow.log_dict({f"{i_:02d}": n for i_, n in enumerate(R.index)}, "improve_i/metric_index.json")
    for f_ in ("imputation_vs_base.csv", "report.json"):
        mlflow.log_artifact(str(OUT / f_), "improve_i")